In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/accepted_2007_to_2018Q4.csv.gz', low_memory=False)

cols = ['loan_amnt', 'term', 'int_rate', 'grade', 'sub_grade',
        'emp_length', 'home_ownership', 'annual_inc', 'purpose', 'dti',
        'issue_d', 'loan_status']
df = df[cols]

# Nos quedamos solo con préstamos de resultado definitivo
df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()

# Creamos el target: 1 = default, 0 = pagado bien
df['default'] = (df['loan_status'] == 'Charged Off').astype(int)

print(f"Total de préstamos para el modelo: {len(df):,}")
print(f"Tasa de default general: {df['default'].mean()*100:.2f}%")
df.head()

Total de préstamos para el modelo: 1,345,310
Tasa de default general: 19.96%


,loan_amnt,term,int_rate,grade,sub_grade,emp_length,home_ownership,annual_inc,purpose,dti,issue_d,loan_status,default
0,3600.0,36 months,13.99,C,C4,10+ years,MORTGAGE,55000.0,debt_consolidation,5.91,Dec-2015,Fully Paid,0
1,24700.0,36 months,11.99,C,C1,10+ years,MORTGAGE,65000.0,small_business,16.06,Dec-2015,Fully Paid,0
2,20000.0,60 months,10.78,B,B4,10+ years,MORTGAGE,63000.0,home_improvement,10.78,Dec-2015,Fully Paid,0
4,10400.0,60 months,22.45,F,F1,3 years,MORTGAGE,104433.0,major_purchase,25.37,Dec-2015,Fully Paid,0
5,11950.0,36 months,13.44,C,C3,4 years,RENT,34000.0,debt_consolidation,10.20,Dec-2015,Fully Paid,0


In [2]:
# Elegimos las variables predictoras (features)
features = ['loan_amnt', 'term', 'int_rate', 'grade', 'emp_length',
            'home_ownership', 'annual_inc', 'purpose', 'dti']

X = df[features].copy()
y = df['default']

# Revisamos qué tipo de dato es cada una
X.dtypes

loan_amnt         float64
term                  str
int_rate          float64
grade                 str
emp_length            str
home_ownership        str
annual_inc        float64
purpose               str
dti               float64
dtype: object

In [3]:
# term viene como " 36 months" o " 60 months" - extraemos solo el número
X['term'] = X['term'].str.extract('(\d+)').astype(float)

# emp_length viene como "10+ years", "< 1 year", "3 years", etc.
# Lo convertimos a número aproximado de años
emp_map = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4,
    '5 years': 5, '6 years': 6, '7 years': 7, '8 years': 8, '9 years': 9,
    '10+ years': 10
}
X['emp_length'] = X['emp_length'].map(emp_map)

X[['term', 'emp_length']].head()

<>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\Joaquin\AppData\Local\Temp\ipykernel_2896\3705840553.py:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  X['term'] = X['term'].str.extract('(\d+)').astype(float)


,term,emp_length
0,36.0,10.0
1,36.0,10.0
2,60.0,10.0
4,60.0,3.0
5,36.0,4.0


In [4]:
# grade tiene un orden natural de riesgo: A (mejor) a G (peor)
grade_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}
X['grade'] = X['grade'].map(grade_map)

X['grade'].head()

0    3
1    3
2    2
4    6
5    3
Name: grade, dtype: int64

In [5]:
X = pd.get_dummies(X, columns=['home_ownership', 'purpose'], drop_first=True)

X.head()

,loan_amnt,term,int_rate,grade,emp_length,annual_inc,dti,home_ownership_MORTGAGE,home_ownership_NONE,home_ownership_OTHER,...,purpose_home_improvement,purpose_house,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding
0,3600.0,36.0,13.99,3,10.0,55000.0,5.91,True,False,False,...,False,False,False,False,False,False,False,False,False,False
1,24700.0,36.0,11.99,3,10.0,65000.0,16.06,True,False,False,...,False,False,False,False,False,False,False,True,False,False
2,20000.0,60.0,10.78,2,10.0,63000.0,10.78,True,False,False,...,True,False,False,False,False,False,False,False,False,False
4,10400.0,60.0,22.45,6,3.0,104433.0,25.37,True,False,False,...,False,False,True,False,False,False,False,False,False,False
5,11950.0,36.0,13.44,3,4.0,34000.0,10.20,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [6]:
# Revisamos si hay valores faltantes
X.isnull().sum()[X.isnull().sum() > 0]

emp_length    78511
dti             374
dtype: int64

In [7]:
# Completamos los valores faltantes con la mediana de cada columna
X['emp_length'] = X['emp_length'].fillna(X['emp_length'].median())
X['dti'] = X['dti'].fillna(X['dti'].median())

# Verificamos que ya no queden nulos
X.isnull().sum().sum()

np.int64(0)

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Dividimos: 80% para entrenar, 20% para evaluar
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Datos de entrenamiento: {len(X_train):,}")
print(f"Datos de evaluación: {len(X_test):,}")
print(f"Tasa de default en train: {y_train.mean()*100:.2f}%")
print(f"Tasa de default en test: {y_test.mean()*100:.2f}%")

Datos de entrenamiento: 1,076,248
Datos de evaluación: 269,062
Tasa de default en train: 19.96%
Tasa de default en test: 19.96%


In [9]:
from sklearn.metrics import roc_auc_score

# Escalamos las variables numéricas (mejora el desempeño de la regresión logística)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Entrenamos el modelo
modelo_logistico = LogisticRegression(max_iter=1000, random_state=42)
modelo_logistico.fit(X_train_scaled, y_train)

# Predecimos probabilidades sobre el set de evaluación (test)
proba_pred = modelo_logistico.predict_proba(X_test_scaled)[:, 1]

# Evaluamos con AUC-ROC
auc = roc_auc_score(y_test, proba_pred)
print(f"AUC-ROC: {auc:.4f}")

AUC-ROC: 0.7014


In [10]:
from xgboost import XGBClassifier

# XGBoost no necesita que los datos estén escalados, así que usamos X_train/X_test originales
modelo_xgb = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42)
modelo_xgb.fit(X_train, y_train)

proba_pred_xgb = modelo_xgb.predict_proba(X_test)[:, 1]

auc_xgb = roc_auc_score(y_test, proba_pred_xgb)
print(f"AUC-ROC (Regresión Logística): {auc:.4f}")
print(f"AUC-ROC (XGBoost): {auc_xgb:.4f}")

AUC-ROC (Regresión Logística): 0.7014
AUC-ROC (XGBoost): 0.7120


In [11]:
importancias = pd.Series(modelo_xgb.feature_importances_, index=X.columns).sort_values(ascending=False)
importancias.head(10)

grade                      0.649263
term                       0.094157
home_ownership_MORTGAGE    0.055433
home_ownership_RENT        0.049142
int_rate                   0.042173
dti                        0.019748
purpose_small_business     0.012260
loan_amnt                  0.012047
annual_inc                 0.010727
emp_length                 0.010606
dtype: float32